# Pre-entrega: Extracción y Almacenamiento Seguro de Datos vía **API**

In [23]:
#Cargo als librerias, como el REQUEST para hacer las peticiones a la API y DOTENV para cargar el archivo .env
import os
import requests
import json
import csv
import pandas as pd
from dotenv import load_dotenv


#Esta línea busca un archivo ".env"
# A partir de aquí, os.getenv podrá leerlas.
load_dotenv()

# os.getenv("VARIABLE", "valor_por_defecto") intenta leer la variable de
# entorno; si no existe, usa el segundo argumento como valor por defecto.

API_URL = os.getenv("POKEAPI_BASE_URL", "https://pokeapi.co/api/v2/pokemon")

# Tiempo máximo (en segundos) donde el requests esperará una respuesta antes
# de lanzar una excepción de tipo Timeout. Evita que el script se quede
# "colgado" para siempre si la API no responde.

REQUEST_TIMEOUT = int(os.getenv("REQUEST_TIMEOUT", "10"))

def obtener_pokemon(pokemon_id):

    try:

        # Realizamos la petición GET
        response = requests.get(
            f"{API_URL}/{pokemon_id}",
            timeout=10
        )

        # Analizamos la respuesta del servidor
        if response.status_code == 200:

            # Convertimos la respuesta JSON
            data = response.json()

            # Seleccionamos solamente los datos que necesitamos
            pokemon = {
                "id": data["id"],
                "nombre": data["name"],
                "altura": data["height"],
                "peso": data["weight"],
                "tipo": data["types"][0]["type"]["name"]
            }

            return pokemon

        elif response.status_code == 404:

            print(f"Error 404: no se encontró el Pokémon {pokemon_id}")

        elif response.status_code == 500:

            print("Error 500: problema en el servidor de la API")

        else:

            print(f"Error HTTP: {response.status_code}")

    except requests.exceptions.ConnectionError:

        print("Error: no se pudo conectar con la API.")

    except requests.exceptions.Timeout:

        print("Error: la solicitud tardó demasiado.")

    except requests.exceptions.RequestException as error:

        print(f"Error en la petición: {error}")

    return None


def guardar_como_json(datos, ruta_archivo):
    # 'with open(...) as archivo' abre el archivo y garantiza que se
    # cierre automáticamente al terminar el bloque (aunque haya un error).
    # "w" = modo escritura. encoding="utf-8" evita problemas con tildes/ñ.
    with open(ruta_archivo, "w", encoding="utf-8") as archivo:
        # json.dump() escribe la estructura de Python directamente al
        # archivo en formato JSON. indent=4 lo deja "bonito" (legible).
        # ensure_ascii=False permite que se guarden tildes/ñ tal cual.
        json.dump(datos, archivo, indent=4, ensure_ascii=False)


def guardar_como_csv(datos, ruta_archivo):
    # Si la lista viene vacía (por ejemplo, porque todas las peticiones
    # fallaron), avisamos y no intentamos escribir un CSV sin columnas.
    if not datos:
        print("[AVISO] No hay datos para guardar en CSV.")
        return

    encabezados = datos[0].keys()

    with open(ruta_archivo, "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=encabezados)
        escritor.writeheader()
        for fila in datos:
            escritor.writerow(fila)



#Aca pruebo si funciona los resultados de la API

resultado = obtener_pokemon(1)
print(resultado)

#el obetener datos me da un diccionario , pero no una lista. Aca lo voy guardando
resultados = []
for i in range(1, 21):  # los primeros 20 pokémon
    pokemon = obtener_pokemon(i)
    if pokemon is not None:
        resultados.append(pokemon)

print(f"Se obtuvieron {len(resultados)} pokémon")

#Aca los guardo en formato Json y CSV apra ver como se ven
guardar_como_json(resultados, "data_extracted.json")
guardar_como_csv(resultados, "data_extracted.csv")

#Aca los leo y muestro como los guardo
pd.read_json("data_extracted.json")
# o
pd.read_csv("data_extracted.csv")

{'id': 1, 'nombre': 'bulbasaur', 'altura': 7, 'peso': 69, 'tipo': 'grass'}
Se obtuvieron 20 pokémon


,id,nombre,altura,peso,tipo
0,1,bulbasaur,7,69,grass
1,2,ivysaur,10,130,grass
2,3,venusaur,20,1000,grass
3,4,charmander,6,85,fire
4,5,charmeleon,11,190,fire
5,6,charizard,17,905,fire
6,7,squirtle,5,90,water
7,8,wartortle,10,225,water
8,9,blastoise,16,855,water
9,10,caterpie,3,29,bug
